# Preprocessing ELM Qualification data

Clara Krämer, May 2025

### Notebook purpose:
1.


##### 1. Set-up and load

In [1]:
# ───────────────────────────────────────────────────────────────────────────────
# STEP 1: STREAM EACH XML INTO ITS OWN PARQUET FILE
# ───────────────────────────────────────────────────────────────────────────────
import os, glob
from lxml import etree
import pyarrow as pa
import pyarrow.parquet as pq

BASE_PATH = "/Users/go82gax/Documents/Projekte/LFS/Analyse/QUAL Daten/aggregated-datasets_LOQ_20250520"
os.chdir(BASE_PATH)

xml_files = sorted(glob.glob("content *.xml"))
OUT_DIR = "parquet_chunks"
os.makedirs(OUT_DIR, exist_ok=True)

RECORD_TAG = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}Description"

for fp in xml_files:
    rows = []
    for _, elem in etree.iterparse(fp, events=("end",), tag=RECORD_TAG):
        rows.append({child.tag: child.text for child in elem})
        elem.clear()
    if rows:
        tbl = pa.Table.from_pylist(rows)
        out_fp = os.path.join(OUT_DIR, os.path.basename(fp).replace(".xml", ".parquet"))
        pq.write_table(tbl, out_fp, compression="snappy")
        print(f"✔ Wrote {len(rows)} rows → {out_fp}")
    else:
        print(f"– Skipped empty {fp}")

✔ Wrote 3351402 rows → parquet_chunks/content 10.parquet
✔ Wrote 96063 rows → parquet_chunks/content 11.parquet
✔ Wrote 735048 rows → parquet_chunks/content 12.parquet
✔ Wrote 1098346 rows → parquet_chunks/content 13.parquet
✔ Wrote 72811 rows → parquet_chunks/content 14.parquet
✔ Wrote 132417 rows → parquet_chunks/content 15.parquet
✔ Wrote 2287 rows → parquet_chunks/content 16.parquet
✔ Wrote 11739 rows → parquet_chunks/content 17.parquet
✔ Wrote 33757 rows → parquet_chunks/content 18.parquet
✔ Wrote 106973 rows → parquet_chunks/content 19.parquet
– Skipped empty content 2.xml
✔ Wrote 924237 rows → parquet_chunks/content 20.parquet
– Skipped empty content 21.xml
✔ Wrote 72109 rows → parquet_chunks/content 22.parquet
– Skipped empty content 23.xml
✔ Wrote 54644 rows → parquet_chunks/content 24.parquet
✔ Wrote 636713 rows → parquet_chunks/content 25.parquet
✔ Wrote 5822 rows → parquet_chunks/content 26.parquet
✔ Wrote 612 rows → parquet_chunks/content 27.parquet
✔ Wrote 415 rows → parq

In [2]:
# ───────────────────────────────────────────────────────────────────────────────
# STEP 2: READ ALL PARQUET CHUNKS AS ONE TABLE & INSPECT
# ───────────────────────────────────────────────────────────────────────────────
import pyarrow.dataset as ds
import pandas as pd

# point to your folder of .parquet files
dataset = ds.dataset("parquet_chunks", format="parquet")

# this reads metadata only; very fast
print(dataset)

# convert to pandas (this is fast because Parquet is columnar & compressed)
df = dataset.to_table().to_pandas()

print("\n── Combined shape ──")
print(df.shape)

print("\n── dtypes & non-null counts ──")
print(df.info())

print("\n── Unique & null counts ──")
stats = pd.concat([df.nunique(dropna=False).rename("unique_count"),
                   df.isna().sum().rename("null_count")], axis=1)
print(stats)

print("\n── Top 5 frequent values per column ──")
for col in df.columns:
    print(f"{col!r}:", df[col].value_counts(dropna=False).head(5), "\n")


── Combined shape ──
(26914461, 2)

── dtypes & non-null counts ──
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26914461 entries, 0 to 26914460
Data columns (total 2 columns):
 #   Column                                             Dtype 
---  ------                                             ----- 
 0   {http://www.w3.org/1999/02/22-rdf-syntax-ns#}type  object
 1   {http://data.europa.eu/snb/model/elm/}noteLiteral  object
dtypes: object(2)
memory usage: 410.7+ MB
None

── Unique & null counts ──
                                                   unique_count  null_count
{http://www.w3.org/1999/02/22-rdf-syntax-ns#}type             1    26914461
{http://data.europa.eu/snb/model/elm/}noteLiteral       1737549    24943362

── Top 5 frequent values per column ──
'{http://www.w3.org/1999/02/22-rdf-syntax-ns#}type': {http://www.w3.org/1999/02/22-rdf-syntax-ns#}type
None    26914461
Name: count, dtype: int64 

'{http://data.europa.eu/snb/model/elm/}noteLiteral': {http://data.europa.eu

##### 2. Inspect

In [3]:
# ───────────────────────────────────────────────────────────────────────────────
# INSPECTION: schema, head, samples, null rates
# ───────────────────────────────────────────────────────────────────────────────
import pandas as pd
import pyarrow.dataset as ds

# 0) Reload via dataset (fast)
dataset = ds.dataset("parquet_chunks", format="parquet")
df = dataset.to_table().to_pandas()

# 1) See full schema
print("Arrow schema:")
print(dataset.schema, "\n")

# 2) Rename columns to strip namespaces for convenience
df = df.rename(columns=lambda col: col.split("}")[-1])

# 3) Quick peek
print("DataFrame shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head(), "\n")

# 4) How many non-null noteLiteral?
print("noteLiteral non-null count:", df['noteLiteral'].notna().sum())
print("noteLiteral null count:   ", df['noteLiteral'].isna().sum(), "\n")

# 5) Sample non-null entries
print("Random sample of noteLiteral (5):")
print(df.loc[df['noteLiteral'].notna(), 'noteLiteral']
         .sample(5, random_state=42)
         .tolist(), "\n")

# 6) Top 10 most common notes
print("Top 10 noteLiteral frequencies:")
print(df['noteLiteral']
        .value_counts(dropna=True)
        .head(10), "\n")

Arrow schema:
{http://www.w3.org/1999/02/22-rdf-syntax-ns#}type: null
{http://data.europa.eu/snb/model/elm/}noteLiteral: string 

DataFrame shape: (26914461, 2)

First 5 rows:
   type                                        noteLiteral
0  None  http://data.europa.eu/europassResource/c615416...
1  None                                               None
2  None  http://data.europa.eu/europassResource/b2f4677...
3  None                                               None
4  None  http://data.europa.eu/europassResource/668aa7e... 

noteLiteral non-null count: 1971099
noteLiteral null count:    24943362 

Random sample of noteLiteral (5):
['http://data.europa.eu/europassResource/00fa517c-a73c-41f7-a023-e1a59819c1cf', 'Wichtige Ausbildungsinhalte:\n\nHöhere Lehranstalten für wirtschaftliche Berufe vermitteln Kenntnisse und Fertigkeiten für Berufe in den Bereichen Wirtschaft, Verwaltung, Tourismus und Ernährung. Neben allgemein bildenden Unterrichtsfächern (Deutsch, Mathematik, Fremdsprachen us

In [4]:
# strip the namespace off for convenience
df = df.rename(columns=lambda c: c.split("}")[-1])

# 1. Distribution of text lengths
df['text_len'] = df['noteLiteral'].fillna("").str.len()
print("Text length summary:")
print(df['text_len'].describe(), "\n")

# 2. Histogram of lengths (binned)
print("Length bins:")
print(pd.cut(df['text_len'],
             bins=[0,50,200,500,1000,5000,10000,50000]).value_counts().sort_index(), "\n")

# 3. Sample long entries (>2000 chars)
longs = df[df['text_len'] > 2000]['noteLiteral']
print("Examples of very long notes (3):")
for txt in longs.sample(3, random_state=1):
    print("-", txt[:200].replace("\n"," "), "…\n")

Text length summary:
count    2.691446e+07
mean     1.044322e+01
std      8.513324e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.457900e+04
Name: text_len, dtype: float64 

Length bins:
text_len
(0, 50]             30436
(50, 200]         1730834
(200, 500]         113842
(500, 1000]         55530
(1000, 5000]        40082
(5000, 10000]         358
(10000, 50000]         17
Name: count, dtype: int64 

Examples of very long notes (3):
- Assessments  range from assignments, to presentations to MCQ Tests.   Assignments: Students shall be required to complete 2,000-word assignments  Presentations: carry out 15-minute presentations as pa …

- Unternehmen und Recht:  Der/die Absolvent/Absolventin ist in der Lage, wichtige rechtliche Grundlagen (Unternehmensrecht, Steuerrecht, Arbeits- und Sozialrecht etc.) für unternehmerische und private E …

- <p>Den här praktikkursen vänder sig till dig som läst kurser inom humaniora och teologi och

In [6]:
# ───────────────────────────────────────────────────────────────────────────────
# INSPECTION: one record's full structure
# ───────────────────────────────────────────────────────────────────────────────

from lxml import etree
import glob, os

# 1) Point to your folder & pick the first file
BASE_PATH = "/Users/go82gax/Documents/Projekte/LFS/Analyse/QUAL Daten/aggregated-datasets_LOQ_20250520"
os.chdir(BASE_PATH)
fp = sorted(glob.glob("content *.xml"))[0]

# 2) Parse and locate the first <rdf:Description>
tree = etree.parse(fp)
ns = {"rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#"}
desc = tree.find(".//rdf:Description", namespaces=ns)

# 3) Print its ID
qual_id = desc.get("{http://www.w3.org/1999/02/22-rdf-syntax-ns#}about")
print("Qualification ID (rdf:about):", qual_id, "\n")

# 4) List all child elements, with attributes and text preview
print("Child elements:")
for child in desc:
    tag = child.tag.split("}")[-1]
    text = child.text or "<None>"
    text_preview = text.replace("\n"," ")[:100] + ("…" if len(text) > 100 else "")
    print(f" • {tag}")
    if child.attrib:
        print(f"     Attributes: {child.attrib}")
    print(f"     Text:      {text_preview}\n")

Qualification ID (rdf:about): http://data.europa.eu/snb/data/europassResource/45826aef-a47d-4b9c-b8ce-77d37c837ce2 

Child elements:
 • type
     Attributes: {'{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource': 'http://data.europa.eu/snb/model/elm/Note'}
     Text:      <None>

 • noteLiteral
     Attributes: {'{http://www.w3.org/XML/1998/namespace}lang': 'en'}
     Text:      http://data.europa.eu/europassResource/c6154165-d760-44a8-9f2d-f9fa0c306574



In [7]:
import re
import pandas as pd

# assume df has a column 'noteLiteral' with all entries (strings or None)
# 1) create a mask for URL-only entries
url_regex = re.compile(r'^\s*https?://')
is_url = df['noteLiteral'].fillna('').str.match(url_regex)

# 2) count URL vs non-URL vs null
total      = len(df)
n_url      = is_url.sum()
n_non_url  = df['noteLiteral'].notna().sum() - n_url
n_null     = df['noteLiteral'].isna().sum()

print(f"Total rows:       {total:,}")
print(f"Null entries:     {n_null:,}")
print(f"URL entries:      {n_url:,} ({n_url/total:.1%} of all rows, {n_url/(n_url+n_non_url):.1%} of non-null)")
print(f"Text entries:     {n_non_url:,} ({n_non_url/total:.1%} of all rows)")

Total rows:       26,914,461
Null entries:     24,943,362
URL entries:      1,629,696 (6.1% of all rows, 82.7% of non-null)
Text entries:     341,403 (1.3% of all rows)
